<a href="https://colab.research.google.com/github/MichalSlowakiewicz/Robot-Control/blob/master/Kopia_notatnika_bml_lab3_distributing_sgd_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Distributing Gradient Descent
Author: Tomasz Kanas

In this class we will learn several methods to distribute Gradient Descent.

We implement the algorithms in PyTorch and run them on the cloud setup we have created last week, using MPI runner.

## Setup cloud environment

You will need the cloud environment we have created last week to complete this class. If you haven't done it yet, firstly finish [this](https://drive.google.com/file/d/1XKDSevaX6SnxbtxlQI5ZjP3dcMdy8HeH/view?usp=sharing) scenario.

To recover the environment you just have to run the following commands in the directory you placed the `simple_deployment.tf` file. Note that the execution may take several minutes.
```sh
. ./setup.sh
terraform apply
python parse-tf-state.py
ansible-playbook -i hosts install_packages.yml
ansible-playbook -i hosts config_ssh.yml
ansible-playbook -i hosts nfs.yml
```

To test the setup copy the master VM IP (the one printed by terraform) to the commands below and run them:

```
export GCP_IP=VM_ID
scp -i .ssh/id_gcp reduce.py $GCP_userID@$GCP_IP:~
ssh -i .ssh/id_gcp $GCP_userID@$GCP_IP "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 reduce.py"
```

## Running MPI tasks from Colab
This step is optional. For your convenience we can setup a way to run code from colab on your cloud VMs.

Firstly, we need to create new ssh key-pair on the colab. Set the value of `GCP_userID` to the username you are using to connect to your VMs, and run the cells below.

In [1]:
%env GCP_userID=m_slowakiewi

env: GCP_userID=m_slowakiewi


In [2]:
!mkdir .ssh
!ssh-keygen -t ed25519 -f .ssh/id_gcp -N '' -C $GCP_userID
!cat .ssh/id_gcp.pub

Generating public/private ed25519 key pair.
Your identification has been saved in .ssh/id_gcp
Your public key has been saved in .ssh/id_gcp.pub
The key fingerprint is:
SHA256:GeCJP+9lbHVtIn50R+I98/YkFuHYFS0pCcYBdRldVCw m_slowakiewi
The key's randomart image is:
+--[ED25519 256]--+
|      . .+=oo= **|
|     o o ...+ E +|
|    . o .    .ooo|
|     .   o   = B |
|      o S   + Oo*|
|       o . o + ==|
|        . = . + +|
|       . +   o +.|
|        .       .|
+----[SHA256]-----+
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIGovRytiF2fi0rNq1M+LVAzKAlxQoPY0sAgpyCrJgEvM m_slowakiewi


The last line of the output is the public key, add it to your cloud metadata:
```
Hamburger (☰) -> Compute Engine -> Metadata
Navigate to 'SSH KEYS' -> 'Edit' -> 'Add Item'
```

NOTE: The files on colab are not persistent, so you will have to repeat those steps every time the colab environment disconnects. To check if you still have the keys, you can use the colab file explorer (folder icon on the left side of the screen). You should see the `.ssh` folder with two key files: `id_gcp` and `id_gcp.pub`.

Now we define the function that will run the `scp` and `ssh` commands from colab. Paste the `key_node` IP from your `hosts` file to the `MASTER_IP` variable and run the cell.

In [7]:
!rm -rf /root/.ssh/known_hosts ~/.ssh/known_hosts

In [8]:
import os
import sys
import argparse
import subprocess

# TODO: paste the key_node IP from your hosts file here
MASTER_IP = "34.69.141.32"

def mpi_run_python(script, mpi_flags):
    guser = os.environ['GCP_userID']

    # copy source file to one of the nodes
    print(f"\n ### copy source code to node {MASTER_IP} ###")
    command = f"scp -i .ssh/id_gcp {script} {guser}@{MASTER_IP}:./"
    print(command)
    p = subprocess.run(command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    p.check_returncode()

    # Exec the mpi program
    print("\n ### Running MPI ###")
    command = f"ssh -i .ssh/id_gcp {guser}@{MASTER_IP} \"mpiexec {mpi_flags} python3 {script}\""
    print(command)
    p = subprocess.run(command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    p.check_returncode()
    # Useful command for debugging
    # os.system("ssh -i {0} {1}@{2} \"{3} --hostfile hostfile_mpi --mca btl_base_warn_component_unused 0 ./{4}\"".format(".ssh/id_gcp", guser, MASTER_IP, nb_ps, execfile))


IMPORTANT: The IPs will be different every time you recreate the VMs, so you need to change the `MASTER_IP` in the script above every time you recreate the VMs.

Although, sometimes when you recreate the VMs they will accidentally get the same IPs what causes problems with SSH security mechanisms (as the public keys of those VMs will change). To be able to troubleshoot such scenario, run the following cell:

In [9]:
!mkdir /root/.ssh/
!touch /root/.ssh/known_hosts

mkdir: cannot create directory ‘/root/.ssh/’: File exists


To run the script above we need to create a file with the script we want to run. It can be done on Colab using the `%%file filename` magic string at the beginning of the cell - then execution will save the cell contents to the file instead of running it. To test this, let's use the example from the previous class:

In [10]:
%%file reduce.py
import os
import time
import torch
import torch.distributed as dist

WORLD_SIZE = int(os.environ['OMPI_COMM_WORLD_SIZE'])
RANK = int(os.environ['OMPI_COMM_WORLD_RANK'])
dist.init_process_group("gloo", rank=RANK, world_size=WORLD_SIZE)

message = torch.tensor(RANK)
print(f"Rank {RANK}: Sending {message.item()}")
dist.reduce(message, 0, dist.ReduceOp.SUM)

if RANK == 0:
  print(f"Rank 0: Sum of all messages is {message.item()}")

Overwriting reduce.py


Now we can run it by executing:

In [11]:
mpi_run_python("reduce.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.69.141.32 ###
scp -i .ssh/id_gcp reduce.py m_slowakiewi@34.69.141.32:./
Host key verification failed.
scp: Connection closed



CalledProcessError: Command 'scp -i .ssh/id_gcp reduce.py m_slowakiewi@34.69.141.32:./' returned non-zero exit status 255.

If you receive an error message: "Host key authentication failed", try those commands (with the IP of the machine the script is sending the code to)

In [ ]:
!ssh-keygen -f "/root/.ssh/known_hosts" -R "35.224.78.203"
!ssh-keyscan 35.224.78.203 | tee -a ~/.ssh/known_hosts

Host 35.224.78.203 not found in /root/.ssh/known_hosts
# 35.224.78.203:22 SSH-2.0-OpenSSH_8.4p1 Debian-5+deb11u3
# 35.224.78.203:22 SSH-2.0-OpenSSH_8.4p1 Debian-5+deb11u3
# 35.224.78.203:22 SSH-2.0-OpenSSH_8.4p1 Debian-5+deb11u3
# 35.224.78.203:22 SSH-2.0-OpenSSH_8.4p1 Debian-5+deb11u3
# 35.224.78.203:22 SSH-2.0-OpenSSH_8.4p1 Debian-5+deb11u3
35.224.78.203 ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAABgQDMDN+cUssjjWsHwGerGM9LbI0/JBCqMLOu2bKD90Fd1L95OdSNtRlYAKvclXxV+LLRvKGhysLKYWw5sFBylybfL/VcmEWEbcth9Z14DgftXPvtFHw+shitof5YV3pyHjn6+j3chdnVaeZDIenW5uzVXXM4cair4RsuQajKa3mM+Ph32DZIhiD+XMcNvti6UBYEgwFXXM99T8sySAgBF4qNUlGZpLo4LTsZOkAdj8z1HL05XnHdkhS4VUgFKmdiLpPG/o1rUNP8fAioKh6P/Br/kTym5YkZ7JjDzvXDTQbS4jSpgR3ddCIRvoptA+u1ZKTZJAkVrJh5WuuDL/6joesRcbCocEeYFKUZsdD2r2+3W5vwA8cYo9KH1OjmXFHcoUs4ufPk8MV555Fy2dCR7GjLqWNM1FRJ4fAN4lrgW6//2/0VIV5JO5E92nWagMWph9eqOKClSwOMtStRtR77HbwDDLq/svSvZtlEhHPKdrZHYVsCEm4KpKMtFjgNPinO7hE=
35.224.78.203 ecdsa-sha2-nistp256 AAAAE2VjZHNhLXNoYTItbmlzdHAyNTYAAAAIbmlzdHAyNTYAAABBBC

If you see
```
bash: warning: setlocale: LC_ALL: cannot change locale en_US.UTF-8)
```
in the output, and it irritates you, run the cell below.


In [ ]:
%env LC_ALL=C

env: LC_ALL=C


## Distributing the Gradient Descent

### Model: multivariate linear regression

For simplicity we will use a multivariate linear regression model, and for data, we will train it on a random data sampled from Normal distribution. Below you have a simple sequential implementation of gradient descent for this model, that we will use as a baseline.

In [ ]:
import numpy as np
import torch

np.set_printoptions(precision=3)
np.set_printoptions(suppress=True)

SEED = 42
np.random.seed(SEED)

TRAINING_DATASET_SIZE = 3000
FEATURES = 6

X_numpy = np.random.multivariate_normal(np.arange(FEATURES), np.eye(FEATURES), TRAINING_DATASET_SIZE)
X_train = torch.tensor(X_numpy).reshape(TRAINING_DATASET_SIZE, FEATURES)
# The hidden coefficients, that model is supposed to learn
COEFFICIENTS = np.array([6, 5, 4, 3, 2, 1], dtype=np.float64)
# Matrix multiplication
y_numpy = X_numpy @ COEFFICIENTS
y_train = torch.tensor(y_numpy)

#Initial solution
w_numpy = np.random.randn(FEATURES)

def forward(w, X):
  return (X @ w).reshape(X.shape[0])

def mse(y, y_est):
  err = y - y_est
  return (err ** 2).mean()

Now we can run simple training loop

In [ ]:
EPOCHS = 200
LEARNING_RATE = 0.01

w = torch.tensor(w_numpy, requires_grad = True)

for epoch in range(EPOCHS):
  y_est = forward(w, X_train)
  loss = mse(y_train, y_est)
  loss.backward()

  with torch.no_grad():
    w -= LEARNING_RATE * w.grad
  w.grad.detach_()
  w.grad.zero_()

w.detach().numpy()

array([5.909, 4.915, 3.942, 2.979, 2.03 , 1.029])

Note, that with the provided parameters, the iteration has *almost* converged - those numbers we can use later to verify, that our distributed algorithms produce exactly the same results.

The remainder of this lab aims at exploring different methods of distributing gradient descent. Actually, as you will quickly realize, the only difficult step is the implementation of the `all_reduce` method, and all the metods we will see, will be in fact different algorithms for `all_reduce`.

### Gradient Accumulation

We will start with very simple concept:
1. Distribute data equally among the nodes.
2. Perform one batch (or epoch in our case, as we have 1 batch per epoch) on each machine.
3. Sum the gradients: use the `reduce` method to compute the sum of gradients in the 0-th node, and then `broadcast`, to broadcast the accumulated gradients to all of the nodes.
4. Repeat this for every batch.

The PyTorch Distributed [documentation](https://pytorch.org/docs/stable/distributed.html) may be handy for those tasks. You can also take a look on the first lab scenario.

Implement this algorithm by extending the code cell below.

NOTE: PyTorch is very sensitive with dimensions and types, so make sure that the types and dimensions of the tensors you send and those you receive match.


In [ ]:
%%file sgd_accumulate.py
import os
import numpy as np
import torch
import torch.distributed as dist

np.set_printoptions(precision=3)
np.set_printoptions(suppress=True)

def forward(w, X):
  return (X @ w).reshape(X.shape[0])

def mse(y, y_est):
  err = y - y_est
  return (err ** 2).mean()

WORLD_SIZE = int(os.environ['OMPI_COMM_WORLD_SIZE'])
RANK = int(os.environ['OMPI_COMM_WORLD_RANK'])
dist.init_process_group("gloo", rank=RANK, world_size=WORLD_SIZE)

SEED = 42
np.random.seed(SEED)

TRAINING_DATASET_SIZE = 3000
FEATURES = 6

assert TRAINING_DATASET_SIZE % WORLD_SIZE == 0, f'{TRAINING_DATASET_SIZE} samples do not split equally between {WORLD_SIZE} workers'
LOCAL_DATASET_SIZE = TRAINING_DATASET_SIZE // WORLD_SIZE

scatter_list = None
w = torch.empty(FEATURES, dtype=torch.float64)
if RANK == 0:
  #Let's suppose that we start with all the data on one machine
  X_numpy = np.random.multivariate_normal(np.arange(FEATURES), np.eye(FEATURES), TRAINING_DATASET_SIZE)
  X_train = torch.tensor(X_numpy).reshape(TRAINING_DATASET_SIZE, FEATURES)
  # The hidden coefficients, that model is supposed to learn
  COEFFICIENTS = np.array([6, 5, 4, 3, 2, 1], dtype=np.float64)
  # Matrix multiplication
  y_numpy = X_numpy @ COEFFICIENTS
  y_train = torch.tensor(y_numpy)

  w_numpy = np.random.randn(FEATURES)
  w = torch.tensor(w_numpy, dtype=torch.float64)
  # Append y to X to send everything in one batch
  data = torch.cat((X_train, y_train[:,None]), dim=1)
  # Divide the data into equal parts that will be sent
  scatter_list = [data[i * LOCAL_DATASET_SIZE : (i + 1) * LOCAL_DATASET_SIZE, :] for i in range(WORLD_SIZE)]

data = torch.empty((LOCAL_DATASET_SIZE, (FEATURES + 1)), dtype=torch.float64) # allocate space for received data - need FEATURES columns for features, and one for y
# Distribute data to all workers
dist.scatter(data, scatter_list, src=0)
# Broadcast the initial state
dist.broadcast(w, src=0)
data = data.reshape(LOCAL_DATASET_SIZE, FEATURES + 1)
X_train = data[:,:-1]
y_train = data[:,-1]

EPOCHS = 200
LEARNING_RATE = 0.01

w.requires_grad = True
loss = torch.tensor([0.0], requires_grad = True)

for epoch in range(EPOCHS):
  #TODO: implement the algorithm
  pass

#At the end every node should have the same coefficients
print(f"rank = {RANK}, data = {np.array2string(w.detach().numpy())}")

Writing sgd_accumulate.py


In [ ]:
mpi_run_python("sgd_accumulate.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 35.224.78.203 ###
scp -i .ssh/id_gcp sgd_accumulate.py t_kanas@35.224.78.203:./


 ### Running MPI ###
ssh -i .ssh/id_gcp t_kanas@35.224.78.203 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 sgd_accumulate.py"
rank = 0, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 2, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 1, data = [5.909 4.915 3.942 2.979 2.03  1.029]



You should see the same state in every node at the end, and the state should match exactly the one in sequential version.

### Parallelizing the communication of the gradients
You might have noticed, that the previous algorithm has problem with scalability: every machine sends its gradient to the machine with rank 0, and then this machine sends accumulated gradient to all machines.

Try to figure it out how we can do this more efficiently before reading the next cell.

#### Algorithm

We will place the machines on the cycle and each machine will send its gradient to the next machine on the cycle (and receive from the previous one).

Then each machine will add its gradient to the received gradient, and in the next iteration will send the sum of the gradients.

We need to perform the same number of iterations as we have machines minus one.

For visualization, see the image below.

<img src="https://i.imgur.com/vHfxzQt.png"/>

NOTE: the picture follows only one of the sums as it travels around the cycle. In the algorithm every node sends and receives in every iteration, so all the nodes - not only `n0` - end up with the total.


Implement this algorithm below. Note that the [`send`](https://mpi4py.readthedocs.io/en/stable/reference/mpi4py.MPI.Comm.html?highlight=Send#mpi4py.MPI.Comm.Send) and [`recv`](https://mpi4py.readthedocs.io/en/stable/reference/mpi4py.MPI.Message.html?highlight=Recv#mpi4py.MPI.Message.Recv) methods are blocking, so to avoid deadlock you need either:
- On even nodes first `send` then `recv` and on the odd nodes first `recv` then `send`.
- Use the asynchronous operations `isend` and `irecv`.
- Use the [`batch_isend_irecv`](https://pytorch.org/docs/stable/distributed.html#torch.distributed.batch_isend_irecv) method.

In [ ]:
%%file sgd_cycle.py
import os
import numpy as np
import torch
import torch.distributed as dist

np.set_printoptions(precision=3)
np.set_printoptions(suppress=True)

def forward(w, X):
  return (X @ w).reshape(X.shape[0])

def mse(y, y_est):
  err = y - y_est
  return (err ** 2).mean()

WORLD_SIZE = int(os.environ['OMPI_COMM_WORLD_SIZE'])
RANK = int(os.environ['OMPI_COMM_WORLD_RANK'])
dist.init_process_group("gloo", rank=RANK, world_size=WORLD_SIZE)

SEED = 42
np.random.seed(SEED)

TRAINING_DATASET_SIZE = 3000
FEATURES = 6

assert TRAINING_DATASET_SIZE % WORLD_SIZE == 0, f'{TRAINING_DATASET_SIZE} samples do not split equally between {WORLD_SIZE} workers'
LOCAL_DATASET_SIZE = TRAINING_DATASET_SIZE // WORLD_SIZE

scatter_list = None
w = torch.empty(FEATURES, dtype=torch.float64)
if RANK == 0:
  #Let's suppose that we start with all the data on one machine
  X_numpy = np.random.multivariate_normal(np.arange(FEATURES), np.eye(FEATURES), TRAINING_DATASET_SIZE)
  X_train = torch.tensor(X_numpy).reshape(TRAINING_DATASET_SIZE, FEATURES)
  # The hidden coefficients, that model is supposed to learn
  COEFFICIENTS = np.array([6, 5, 4, 3, 2, 1], dtype=np.float64)
  # Matrix multiplication
  y_numpy = X_numpy @ COEFFICIENTS
  y_train = torch.tensor(y_numpy)

  w_numpy = np.random.randn(FEATURES)
  w = torch.tensor(w_numpy, dtype=torch.float64)
  # Append y to X to send everything in one batch
  data = torch.cat((X_train, y_train[:,None]), dim=1)
  # Divide the data into equal parts that will be sent
  scatter_list = [data[i * LOCAL_DATASET_SIZE : (i + 1) * LOCAL_DATASET_SIZE, :] for i in range(WORLD_SIZE)]

data = torch.empty((LOCAL_DATASET_SIZE, (FEATURES + 1)), dtype=torch.float64) # allocate space for received data - need FEATURES columns for features, and one for y
# Distribute data to all workers
dist.scatter(data, scatter_list, src=0)
# Broadcast the initial state
dist.broadcast(w, src=0)
data = data.reshape(LOCAL_DATASET_SIZE, FEATURES + 1)
X_train = data[:,:-1]
y_train = data[:,-1]

EPOCHS = 200
LEARNING_RATE = 0.01

w.requires_grad = True
loss = torch.tensor([0.0], requires_grad = True)

for epoch in range(EPOCHS):
  #TODO: implement the algorithm
  pass

#At the end every node should have the same coefficients
print(f"rank = {RANK}, data = {np.array2string(w.detach().numpy())}")

Overwriting sgd_cycle.py


In [ ]:
mpi_run_python("sgd_cycle.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 35.224.78.203 ###
scp -i .ssh/id_gcp sgd_cycle.py t_kanas@35.224.78.203:./


 ### Running MPI ###
ssh -i .ssh/id_gcp t_kanas@35.224.78.203 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 sgd_cycle.py"
rank = 2, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 0, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 1, data = [5.909 4.915 3.942 2.979 2.03  1.029]



Again, verify that all nodes have the same coefficients at the end, and those coefficients are equal to the ones produced by the sequential solution.

### Horovod algorithm

Horovod algorithm goes one step further than the previous algorithm. In large models, sending whole gradient in one round may be too much for you bandwidth capacity. In the Horovod algorithm, in one round we will send only one chunk of the gradient. Because of that, we will actually need two full cycles of communication to distribute all the gradients among all the nodes.

The algorithm looks like this:
1. Compute forward and backward pass on the local part of the data.
2. Split the gradient into the same number of chunks, you have workers in the network.
3. In `i`-th iteration the worker with rank `RANK` sends the accumulated chunk number `RANK - i - 1` (the one it received in the previous iteration) to the next node. We do `WORLD_SIZE - 1` of such iterations.
4. After `WORLD_SIZE - 1` such iterations, each node has one fully aggregated data chunk.
5. Now we do another round on the cycle, to distribute the fully aggregated data chunks among the nodes.

See the animation below for visualization.

In [ ]:
%%html
<video  width="960" height="720" autoplay loop muted playsinline controls>
    <source src="https://i.imgur.com/IV6jBwL.mp4" type="video/mp4">
</video>

Implement this algorithm below.

In [ ]:
%%file horovod.py
import os
import numpy as np
import torch
import torch.distributed as dist

np.set_printoptions(precision=3)
np.set_printoptions(suppress=True)

def forward(w, X):
  return (X @ w).reshape(X.shape[0])

def mse(y, y_est):
  err = y - y_est
  return (err ** 2).mean()

WORLD_SIZE = int(os.environ['OMPI_COMM_WORLD_SIZE'])
RANK = int(os.environ['OMPI_COMM_WORLD_RANK'])
dist.init_process_group("gloo", rank=RANK, world_size=WORLD_SIZE)

SEED = 42
np.random.seed(SEED)

TRAINING_DATASET_SIZE = 3000
FEATURES = 6

assert TRAINING_DATASET_SIZE % WORLD_SIZE == 0, f'{TRAINING_DATASET_SIZE} samples do not split equally between {WORLD_SIZE} workers'
LOCAL_DATASET_SIZE = TRAINING_DATASET_SIZE // WORLD_SIZE
assert FEATURES % WORLD_SIZE == 0, f'{FEATURES} features do not split into {WORLD_SIZE} equal chunks'

scatter_list = None
w = torch.empty(FEATURES, dtype=torch.float64)
if RANK == 0:
  #Let's suppose that we start with all the data on one machine
  X_numpy = np.random.multivariate_normal(np.arange(FEATURES), np.eye(FEATURES), TRAINING_DATASET_SIZE)
  X_train = torch.tensor(X_numpy).reshape(TRAINING_DATASET_SIZE, FEATURES)
  # The hidden coefficients, that model is supposed to learn
  COEFFICIENTS = np.array([6, 5, 4, 3, 2, 1], dtype=np.float64)
  # Matrix multiplication
  y_numpy = X_numpy @ COEFFICIENTS
  y_train = torch.tensor(y_numpy)

  w_numpy = np.random.randn(FEATURES)
  w = torch.tensor(w_numpy, dtype=torch.float64)
  # Append y to X to send everything in one batch
  data = torch.cat((X_train, y_train[:,None]), dim=1)
  # Divide the data into equal parts that will be sent
  scatter_list = [data[i * LOCAL_DATASET_SIZE : (i + 1) * LOCAL_DATASET_SIZE, :] for i in range(WORLD_SIZE)]

data = torch.empty((LOCAL_DATASET_SIZE, (FEATURES + 1)), dtype=torch.float64) # allocate space for received data - need FEATURES columns for features, and one for y
# Distribute data to all workers
dist.scatter(data, scatter_list, src=0)
# Broadcast the initial state
dist.broadcast(w, src=0)
data = data.reshape(LOCAL_DATASET_SIZE, FEATURES + 1)
X_train = data[:,:-1]
y_train = data[:,-1]

EPOCHS = 200
LEARNING_RATE = 0.01

w.requires_grad = True
loss = torch.tensor([0.0], requires_grad = True)

for epoch in range(EPOCHS):
  #TODO: implement the algorithm
  pass

#At the end every node should have the same coefficients
print(f"rank = {RANK}, data = {np.array2string(w.detach().numpy())}")

Overwriting horovod.py


In [ ]:
mpi_run_python("horovod.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 35.224.78.203 ###
scp -i .ssh/id_gcp horovod.py t_kanas@35.224.78.203:./


 ### Running MPI ###
ssh -i .ssh/id_gcp t_kanas@35.224.78.203 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 horovod.py"
rank = 0, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 2, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 1, data = [5.909 4.915 3.942 2.979 2.03  1.029]



## Exercise

Measure the training time for several (at least 4) different numbers of features for all 3 presented algorithms. What can you deduce from the results?

# IMPORTANT: Remove created resources at the end of the session
```
terraform destroy
```

**And don't forget to enter 'yes'!**

We can easily recreate them now, so nothing is lost upon deletion!